# Compositional Data Quality Calculus (CDQC) — Final Experiments
## Predicting Multi-Dimensional Quality Effects in ML Pipelines

## 1. Environment Setup

In [1]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["ucimlrepo", "scikit-learn", "pandas", "numpy", "matplotlib",
            "seaborn", "xgboost", "scipy", "statsmodels", "tqdm",
            "tabulate", "scikit-posthocs"]:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        install(pkg)

print("All packages ready")

All packages ready


In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for VS Code
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from itertools import combinations
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, field
from tqdm.auto import tqdm
import time, json, os, warnings, copy, gc, hashlib
from tabulate import tabulate
import scikit_posthocs as sp
warnings.filterwarnings('ignore')

from sklearn.model_selection import cross_validate, StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.base import clone
from sklearn.metrics import (make_scorer, balanced_accuracy_score,
                             f1_score, matthews_corrcoef,
                             mean_squared_error, mean_absolute_error)

# ═══════════════════════════════════════════════════════════════
# GLOBAL CONFIGURATION
# ═══════════════════════════════════════════════════════════════
RANDOM_SEED   = 42
MAX_SAMPLES   = 5000
N_FOLDS       = 5
N_SEEDS_AB    = 3      # repeat seeds for Phases A & B
N_SEEDS_C     = 2      # repeat seeds for Phase C (runtime trade-off)
CHECKPOINT_DIR = './cdqc_final_checkpoints'
FIGURE_DIR     = './cdqc_final_figures'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(FIGURE_DIR, exist_ok=True)

# ── Metrics ──
METRICS = {
    'accuracy':          'accuracy',
    'balanced_accuracy': 'balanced_accuracy',
    'macro_f1':          make_scorer(f1_score, average='macro', zero_division=0),
    'mcc':               make_scorer(matthews_corrcoef),
}
REPORT_METRICS = list(METRICS.keys())

# ── Interaction thresholds (R2-D5) ──
INTERACTION_THRESHOLDS = [0.005, 0.01, 0.02, 0.05]
DEFAULT_THRESHOLD = 0.01

# ── GPU detection for XGBoost ──
try:
    import torch
    HAS_CUDA = torch.cuda.is_available()
    if HAS_CUDA:
        print(f"  CUDA detected: {torch.cuda.get_device_name(0)}")
except ImportError:
    HAS_CUDA = False

# ── Plotting ──
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300, 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 12,
    'figure.titlesize': 14, 'font.family': 'serif',
})

np.random.seed(RANDOM_SEED)
print(f"  Seed={RANDOM_SEED}  Folds={N_FOLDS}  MaxSamples={MAX_SAMPLES}")
print(f"  Metrics: {REPORT_METRICS}")
print(f"  GPU XGBoost: {HAS_CUDA}")
print(f"  Checkpoints: {CHECKPOINT_DIR}/")

c:\Users\Gulzar\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  Seed=42  Folds=5  MaxSamples=5000
  Metrics: ['accuracy', 'balanced_accuracy', 'macro_f1', 'mcc']
  GPU XGBoost: False
  Checkpoints: ./cdqc_final_checkpoints/


## 2. Utilities (Checkpointing, Bootstrap, Timing)

In [3]:
def save_checkpoint(data, name):
    """Save DataFrame or dict to checkpoint."""
    path = f"{CHECKPOINT_DIR}/{name}.csv"
    if isinstance(data, pd.DataFrame):
        data.to_csv(path, index=False)
    elif isinstance(data, list):
        pd.DataFrame(data).to_csv(path, index=False)
    return path

def load_checkpoint(name):
    path = f"{CHECKPOINT_DIR}/{name}.csv"
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"  Loaded checkpoint: {name} ({len(df)} rows)")
        return df
    return None

def append_checkpoint(records, name):
    """Append new records to existing checkpoint (for crash recovery)."""
    path = f"{CHECKPOINT_DIR}/{name}.csv"
    new_df = pd.DataFrame(records) if isinstance(records, list) else records
    if os.path.exists(path):
        existing = pd.read_csv(path)
        combined = pd.concat([existing, new_df], ignore_index=True)
    else:
        combined = new_df
    combined.to_csv(path, index=False)
    return combined

def get_completed_keys(name, key_cols):
    """Get set of already-completed experiment keys for resumption."""
    path = f"{CHECKPOINT_DIR}/{name}.csv"
    if os.path.exists(path):
        df = pd.read_csv(path)
        return set(df[key_cols].apply(tuple, axis=1).tolist())
    return set()

def bootstrap_ci(values, n_boot=1000, ci=0.95, seed=RANDOM_SEED):
    rng = np.random.RandomState(seed)
    boot_means = [np.mean(rng.choice(values, len(values), replace=True))
                  for _ in range(n_boot)]
    alpha = (1 - ci) / 2
    return np.percentile(boot_means, [100*alpha, 100*(1-alpha)])

class Timer:
    def __enter__(self):
        self.start = time.perf_counter(); return self
    def __exit__(self, *a):
        self.elapsed = time.perf_counter() - self.start
        m, s = divmod(self.elapsed, 60)
        print(f"  Elapsed: {int(m)}m {s:.1f}s")

def save_fig(fig, name):
    for ext in ['pdf', 'png']:
        fig.savefig(f"{FIGURE_DIR}/{name}.{ext}", bbox_inches='tight', dpi=300)
    plt.close(fig)
    print(f"  Saved figure: {name}")

print("Utilities ready")

Utilities ready


## 3. Dataset Loading

Five benchmark datasets spanning diverse domains:

| Dataset | Domain | Why Selected |
|---------|--------|-------------|
| **Spambase** | Cybersecurity | Replaces Adult (encoding issues); 4,601 samples, 57 features, well-cited |
| **Bank** | Marketing | Large-scale, substantial imbalance (7.5:1) |
| **Wine** | Chemistry | Small-sample, multi-class (3 classes) |
| **Breast Cancer** | Medical | Medical diagnosis, slight imbalance |
| **Digits** | Vision | High-dimensional (64 features), balanced |

In [4]:
from ucimlrepo import fetch_ucirepo
from sklearn.datasets import load_digits, load_breast_cancer

@dataclass
class Dataset:
    name: str
    X: np.ndarray
    y: np.ndarray
    domain: str = ""
    n_samples: int = field(init=False)
    n_features: int = field(init=False)
    n_classes: int = field(init=False)

    def __post_init__(self):
        self.n_samples, self.n_features = self.X.shape
        self.n_classes = len(np.unique(self.y))

def load_uci(dataset_id, name, domain, max_samples=MAX_SAMPLES):
    print(f"  Loading {name} (UCI #{dataset_id})...", end=" ")
    data = fetch_ucirepo(id=dataset_id)
    X = data.data.features.copy()
    y = data.data.targets.copy()
    if isinstance(y, pd.DataFrame):
        y = y.iloc[:, 0]
    y = LabelEncoder().fit_transform(y.values.ravel())

    # Handle mixed types: one-hot encode categoricals
    if isinstance(X, pd.DataFrame):
        cat_cols = X.select_dtypes(include=['object', 'category']).columns
        if len(cat_cols) > 0:
            X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
        X = X.apply(pd.to_numeric, errors='coerce')

    X = SimpleImputer(strategy='median').fit_transform(X)
    X = StandardScaler().fit_transform(X)

    if len(X) > max_samples:
        rng = np.random.RandomState(RANDOM_SEED)
        _, X, _, y = train_test_split(X, y, test_size=max_samples,
                                       stratify=y, random_state=RANDOM_SEED)
        print(f"(subsampled to {max_samples})", end=" ")
    print(f"shape={X.shape}  classes={len(np.unique(y))}")
    return Dataset(name=name, X=X, y=y, domain=domain)

def load_sklearn_ds(loader_fn, name, domain, force_binary=False):
    print(f"  Loading {name}...", end=" ")
    data = loader_fn()
    X, y = data.data, data.target
    if force_binary and len(np.unique(y)) > 2:
        # Digits: 0-4 vs 5-9
        y = (y >= 5).astype(int)
    X = StandardScaler().fit_transform(X)
    print(f"shape={X.shape}  classes={len(np.unique(y))}")
    return Dataset(name=name, X=X, y=y, domain=domain)

# ═══════════════════════════════════════════════════════════════
# LOAD ALL DATASETS
# ═══════════════════════════════════════════════════════════════
print("="*60)
print("  LOADING DATASETS")
print("="*60)

DATASETS = {}
try:
    DATASETS['spambase'] = load_uci(94, 'Spambase', 'Cybersecurity')
except Exception as e:
    print(f"  Spambase failed: {e}")

try:
    DATASETS['bank'] = load_uci(222, 'Bank', 'Marketing')
except Exception as e:
    print(f"  Bank failed: {e}")

try:
    DATASETS['wine'] = load_uci(109, 'Wine', 'Chemistry')
except Exception as e:
    print(f"  Wine failed: {e}")

DATASETS['breast_cancer'] = load_sklearn_ds(load_breast_cancer, 'BreastCancer', 'Medical')
DATASETS['digits'] = load_sklearn_ds(load_digits, 'Digits', 'Vision', force_binary=True)

# ── Summary ──
rows = []
for k, ds in DATASETS.items():
    cls_counts = np.bincount(ds.y)
    ratio = max(cls_counts) / (min(cls_counts) + 1e-9)
    rows.append([ds.name, ds.n_samples, ds.n_features, ds.n_classes,
                 ds.domain, f"{ratio:.1f}:1"])
print()
print(tabulate(rows,
    headers=['Dataset','Samples','Features','Classes','Domain','Imbalance'],
    tablefmt='grid'))
print(f"\nLoaded {len(DATASETS)} datasets")

  LOADING DATASETS
  Loading Spambase (UCI #94)... shape=(4601, 57)  classes=2
  Loading Bank (UCI #222)... (subsampled to 5000) shape=(5000, 38)  classes=2
  Loading Wine (UCI #109)... shape=(178, 13)  classes=3
  Loading BreastCancer... shape=(569, 30)  classes=2
  Loading Digits... shape=(1797, 64)  classes=2

+--------------+-----------+------------+-----------+---------------+-------------+
| Dataset      |   Samples |   Features |   Classes | Domain        | Imbalance   |
+==============+===========+============+===========+===============+=============+
| Spambase     |      4601 |         57 |         2 | Cybersecurity | 1.5:1       |
+--------------+-----------+------------+-----------+---------------+-------------+
| Bank         |      5000 |         38 |         2 | Marketing     | 7.5:1       |
+--------------+-----------+------------+-----------+---------------+-------------+
| Wine         |       178 |         13 |         3 | Chemistry     | 1.5:1       |
+------------

## 4. Corruption Functions (All Bugs Fixed)

| Fix | Detail |
|-----|--------|
| **Label noise** | Samples replacement from Y\\{yᵢ} exclusively |
| **Feature noise** | Scales by per-feature σⱼ |
| **Class imbalance** | severity=0 → balanced, severity=0.8 → 80% minority removed; multi-class safe |
| **Missing features** | MCAR with median imputation |

In [5]:
def corrupt_labels(X, y, severity, seed=None):
    """Flip each label to a uniformly sampled INCORRECT class with probability=severity."""
    if severity <= 0:
        return X.copy(), y.copy()
    rng = np.random.RandomState(seed)
    y_new = y.copy()
    classes = np.unique(y)
    n = len(y)
    flip_mask = rng.random(n) < severity
    for i in np.where(flip_mask)[0]:
        wrong_classes = classes[classes != y[i]]
        y_new[i] = rng.choice(wrong_classes)
    return X.copy(), y_new

def corrupt_missing(X, y, severity, seed=None):
    """Set each feature value to NaN with prob=severity (MCAR), impute with column median."""
    if severity <= 0:
        return X.copy(), y.copy()
    rng = np.random.RandomState(seed)
    X_new = X.copy()
    mask = rng.random(X.shape) < severity
    col_medians = np.nanmedian(X, axis=0)
    for col in range(X.shape[1]):
        X_new[mask[:, col], col] = col_medians[col]
    return X_new, y.copy()

def corrupt_features(X, y, severity, seed=None):
    """Add Gaussian noise: x_ij += N(0, severity * sigma_j)."""
    if severity <= 0:
        return X.copy(), y.copy()
    rng = np.random.RandomState(seed)
    sigma_j = np.std(X, axis=0) + 1e-10
    noise = rng.normal(0, 1, X.shape) * (severity * sigma_j)
    return X + noise, y.copy()

def corrupt_imbalance(X, y, severity, seed=None):
    """Remove severity fraction of each minority class's samples."""
    if severity <= 0:
        return X.copy(), y.copy()
    rng = np.random.RandomState(seed)
    classes, counts = np.unique(y, return_counts=True)
    majority_class = classes[np.argmax(counts)]
    keep_idx = []
    for cls in classes:
        cls_idx = np.where(y == cls)[0]
        if cls == majority_class:
            keep_idx.append(cls_idx)
        else:
            n_keep = max(2, int(len(cls_idx) * (1 - severity)))
            keep_idx.append(rng.choice(cls_idx, n_keep, replace=False))
    selected = np.concatenate(keep_idx)
    rng.shuffle(selected)
    return X[selected], y[selected]

# ═══════════════════════════════════════════════════════════════
# DIMENSION CONFIGURATION — single source of truth
# ═══════════════════════════════════════════════════════════════
DIMENSIONS = {
    'label_noise': {
        'fn': corrupt_labels,
        'levels_single': [0.0, 0.10, 0.20, 0.35, 0.50],
        'levels_pair':   [0.15, 0.35],
        'name': 'Label Noise', 'short': 'LN',
    },
    'missing': {
        'fn': corrupt_missing,
        'levels_single': [0.0, 0.05, 0.10, 0.20, 0.30],
        'levels_pair':   [0.10, 0.25],
        'name': 'Missing Features', 'short': 'MF',
    },
    'feature_noise': {
        'fn': corrupt_features,
        'levels_single': [0.0, 0.10, 0.20, 0.35, 0.50],
        'levels_pair':   [0.20, 0.45],
        'name': 'Feature Noise', 'short': 'FN',
    },
    'imbalance': {
        'fn': corrupt_imbalance,
        'levels_single': [0.0, 0.20, 0.40, 0.60, 0.80],
        'levels_pair':   [0.30, 0.60],
        'name': 'Class Imbalance', 'short': 'CI',
    },
}

print("Corruption functions ready")
for k, d in DIMENSIONS.items():
    print(f"  {d['name']:18s} single={d['levels_single']}  pair={d['levels_pair']}")

Corruption functions ready
  Label Noise        single=[0.0, 0.1, 0.2, 0.35, 0.5]  pair=[0.15, 0.35]
  Missing Features   single=[0.0, 0.05, 0.1, 0.2, 0.3]  pair=[0.1, 0.25]
  Feature Noise      single=[0.0, 0.1, 0.2, 0.35, 0.5]  pair=[0.2, 0.45]
  Class Imbalance    single=[0.0, 0.2, 0.4, 0.6, 0.8]  pair=[0.3, 0.6]


## 5. ML Algorithms

In [6]:
# XGBoost GPU config
xgb_params = dict(
    n_estimators=100, max_depth=5, learning_rate=0.1,
    use_label_encoder=False, eval_metric='mlogloss', verbosity=0,
    random_state=RANDOM_SEED
)
if HAS_CUDA:
    xgb_params['tree_method'] = 'gpu_hist'
    xgb_params['predictor'] = 'gpu_predictor'

ALGORITHMS = {
    'LR':  {'model': LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs',
                                         random_state=RANDOM_SEED), 'family': 'Linear'},
    'RF':  {'model': RandomForestClassifier(n_estimators=100, max_depth=12,
                                             n_jobs=-1, random_state=RANDOM_SEED), 'family': 'Ensemble'},
    'XGB': {'model': XGBClassifier(**xgb_params), 'family': 'Boosting'},
    'SVM': {'model': SVC(C=1.0, kernel='rbf', gamma='scale',
                          random_state=RANDOM_SEED), 'family': 'Kernel'},
    'MLP': {'model': MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300,
                                    early_stopping=True, random_state=RANDOM_SEED), 'family': 'Neural'},
    'GB':  {'model': GradientBoostingClassifier(n_estimators=100, max_depth=4,
                                                learning_rate=0.1, random_state=RANDOM_SEED), 'family': 'Boosting'},
}

for k, v in ALGORITHMS.items():
    print(f"  {k:4s} ({v['family']})")
print(f"\n{len(ALGORITHMS)} algorithms configured")

  LR   (Linear)
  RF   (Ensemble)
  XGB  (Boosting)
  SVM  (Kernel)
  MLP  (Neural)
  GB   (Boosting)

6 algorithms configured


## 6. Multi-Metric Experiment Runner

In [7]:
def run_experiment(dataset, algo_key, algo_info, dim_severities, seed=RANDOM_SEED):
    """Run a single experiment, return dict with all 4 metrics or None on failure."""
    X, y = dataset.X.copy(), dataset.y.copy()

    # Apply corruptions
    for dim_name, severity in dim_severities.items():
        if severity > 0:
            X, y = DIMENSIONS[dim_name]['fn'](X, y, severity, seed=seed)

    # Guard
    if len(y) < N_FOLDS * 3 or len(np.unique(y)) < 2:
        return None

    try:
        cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        cv_results = cross_validate(
            clone(algo_info['model']), X, y,
            cv=cv, scoring=METRICS, n_jobs=-1, return_train_score=False
        )
    except Exception:
        return None

    result = {
        'dataset': dataset.name, 'algorithm': algo_key,
        'algo_family': algo_info['family'],
        **dim_severities, 'n_samples': len(y), 'seed': seed,
    }
    for m in REPORT_METRICS:
        scores = cv_results[f'test_{m}']
        result[m] = float(np.mean(scores))
        result[f'{m}_std'] = float(np.std(scores))

    return result

print("Experiment runner ready")

Experiment runner ready


## 7. Phase A — Clean Baselines

In [8]:
baselines = load_checkpoint('phase_a_baselines')

if baselines is None:
    print("="*60)
    print("  PHASE A: CLEAN BASELINES")
    print("="*60)
    with Timer():
        results = []
        clean_profile = {d: 0 for d in DIMENSIONS}
        for ds_key, dataset in DATASETS.items():
            for algo_key, algo_info in ALGORITHMS.items():
                for s in range(N_SEEDS_AB):
                    r = run_experiment(dataset, algo_key, algo_info,
                                      clean_profile, seed=RANDOM_SEED+s)
                    if r:
                        results.append(r)
        baselines = pd.DataFrame(results)
        save_checkpoint(baselines, 'phase_a_baselines')

# ── Display ──
for m in ['accuracy', 'balanced_accuracy']:
    pivot = baselines.groupby(['dataset','algorithm'])[m].mean().unstack()
    print(f"\nClean-data {m}:")
    print(pivot.round(4).to_string())

print(f"\nPhase A: {len(baselines)} runs")

  PHASE A: CLEAN BASELINES
  Elapsed: 0m 37.4s

Clean-data accuracy:
algorithm         GB      LR     MLP      RF     SVM     XGB
dataset                                                     
Bank          0.8944  0.8961  0.8910  0.8959  0.8902  0.8957
BreastCancer  0.9561  0.9754  0.9467  0.9614  0.9754  0.9672
Digits        0.9672  0.8928  0.9636  0.9777  0.9807  0.9701
Spambase      0.9477  0.9252  0.9370  0.9448  0.9329  0.9504
Wine          0.9308  0.9851  0.9195  0.9832  0.9832  0.9570

Clean-data balanced_accuracy:
algorithm         GB      LR     MLP      RF     SVM     XGB
dataset                                                     
Bank          0.6659  0.6416  0.6192  0.6035  0.5884  0.6644
BreastCancer  0.9510  0.9709  0.9380  0.9575  0.9721  0.9624
Digits        0.9672  0.8928  0.9637  0.9777  0.9807  0.9701
Spambase      0.9430  0.9183  0.9327  0.9374  0.9263  0.9461
Wine          0.9289  0.9858  0.9237  0.9854  0.9820  0.9589

Phase A: 90 runs


## 8. Phase B — Single-Dimension Experiments

**Crash-safe**: saves after each (dimension, severity) block.
If interrupted, re-run this cell — it will skip completed blocks.

In [9]:
CP_NAME_B = 'phase_b_single_dim'
single_dim = load_checkpoint(CP_NAME_B)

if single_dim is None or len(single_dim) == 0:
    # Fresh start or empty checkpoint
    existing_keys = set()
    all_results = []
else:
    all_results = single_dim.to_dict('records')
    # Build set of completed (dim, severity, dataset, algorithm, seed) tuples
    existing_keys = set()
    for r in all_results:
        existing_keys.add((r.get('active_dim',''), r.get('severity',0),
                           r['dataset'], r['algorithm'], r.get('seed',0)))

total_expected = sum(len(d['levels_single'])-1 for d in DIMENSIONS.values()) \
                 * len(DATASETS) * len(ALGORITHMS) * N_SEEDS_AB
remaining = total_expected - len(existing_keys)

if remaining > 0:
    print("="*60)
    print(f"  PHASE B: SINGLE-DIMENSION ({remaining} remaining of {total_expected})")
    print("="*60)

    with Timer():
        batch = []
        batch_count = 0

        with tqdm(total=remaining, desc="Phase B") as pbar:
            for dim_name, dim_info in DIMENSIONS.items():
                for severity in dim_info['levels_single']:
                    if severity == 0:
                        continue
                    for ds_key, dataset in DATASETS.items():
                        for algo_key, algo_info in ALGORITHMS.items():
                            for s in range(N_SEEDS_AB):
                                key = (dim_name, severity, dataset.name,
                                       algo_key, RANDOM_SEED+s)
                                if key in existing_keys:
                                    continue

                                profile = {d: 0 for d in DIMENSIONS}
                                profile[dim_name] = severity
                                r = run_experiment(dataset, algo_key, algo_info,
                                                  profile, seed=RANDOM_SEED+s)
                                if r:
                                    r['active_dim'] = dim_name
                                    r['severity'] = severity
                                    batch.append(r)
                                    all_results.append(r)
                                pbar.update(1)
                                batch_count += 1

                    # Save after each (dim, severity) block
                    if batch:
                        append_checkpoint(batch, CP_NAME_B)
                        batch = []
                        gc.collect()

        # Final save
        if batch:
            append_checkpoint(batch, CP_NAME_B)

    single_dim = pd.DataFrame(all_results)
    save_checkpoint(single_dim, CP_NAME_B)
else:
    single_dim = pd.DataFrame(all_results)

print(f"\nPhase B complete: {len(single_dim)} experiments")

  PHASE B: SINGLE-DIMENSION (1440 remaining of 1440)


Phase B: 100%|██████████| 1440/1440 [09:03<00:00,  2.65it/s]

  Elapsed: 9m 3.4s

Phase B complete: 1440 experiments


In [10]:
def add_degradation(results_df, baselines_df, metrics=REPORT_METRICS):
    """Compute degradation = baseline - corrupted for every metric."""
    df = results_df.copy()
    bl = baselines_df.groupby(['dataset','algorithm'])[metrics].mean()
    for metric in metrics:
        deg_col = f'deg_{metric}'
        df[deg_col] = 0.0
        for idx, row in df.iterrows():
            key = (row['dataset'], row['algorithm'])
            if key in bl.index:
                df.at[idx, deg_col] = bl.loc[key, metric] - row[metric]
    df['degradation'] = df['deg_accuracy']
    return df

single_dim = add_degradation(single_dim, baselines)
print("Degradation columns added")

Degradation columns added


## 9. Sensitivity Coefficients (Multi-Metric, Bootstrap CIs)

In [11]:
def estimate_alpha_coefficients(results_df, n_boot=1000):
    coefficients = []
    for algo in sorted(results_df['algorithm'].unique()):
        for dim in DIMENSIONS.keys():
            mask = (results_df['algorithm'] == algo) & (results_df['active_dim'] == dim)
            sub = results_df[mask]
            if len(sub) < 2:
                continue
            for metric in REPORT_METRICS:
                deg_col = f'deg_{metric}'
                agg = sub.groupby('severity')[deg_col].mean().reset_index()
                x, y_mean = agg['severity'].values, agg[deg_col].values

                # OLS through origin
                alpha = np.sum(x * y_mean) / (np.sum(x**2) + 1e-12)
                y_pred = alpha * x
                ss_res = np.sum((y_mean - y_pred)**2)
                ss_tot = np.sum((y_mean - np.mean(y_mean))**2) + 1e-12
                r2 = max(0, 1 - ss_res / ss_tot)

                # Bootstrap CI
                raw_x = sub['severity'].values
                raw_y = sub[deg_col].values
                rng = np.random.RandomState(RANDOM_SEED)
                boot_alphas = []
                for _ in range(n_boot):
                    idx = rng.choice(len(raw_x), len(raw_x), replace=True)
                    ba = np.sum(raw_x[idx]*raw_y[idx]) / (np.sum(raw_x[idx]**2)+1e-12)
                    boot_alphas.append(ba)
                ci_lo, ci_hi = np.percentile(boot_alphas, [2.5, 97.5])

                coefficients.append({
                    'algorithm': algo, 'dimension': dim,
                    'dim_name': DIMENSIONS[dim]['name'],
                    'dim_short': DIMENSIONS[dim]['short'],
                    'metric': metric, 'alpha': alpha,
                    'alpha_ci_lo': ci_lo, 'alpha_ci_hi': ci_hi,
                    'r_squared': r2, 'n_obs': len(raw_x),
                })
    return pd.DataFrame(coefficients)

alpha_df = estimate_alpha_coefficients(single_dim)
save_checkpoint(alpha_df, 'alpha_coefficients')

for metric in REPORT_METRICS:
    sub = alpha_df[alpha_df['metric'] == metric]
    pivot = sub.pivot_table(index='dim_name', columns='algorithm', values='alpha', aggfunc='mean')
    pivot['Mean'] = pivot.mean(axis=1)
    print(f"\n{'='*70}")
    print(f"  alpha COEFFICIENTS - metric: {metric}")
    print(f"{'='*70}")
    print(pivot.round(4).to_string())

print("\nAlpha estimation complete with bootstrap CIs")


  alpha COEFFICIENTS - metric: accuracy
algorithm             GB      LR     MLP      RF     SVM     XGB    Mean
dim_name                                                                
Class Imbalance  -0.0172 -0.0252  0.0441 -0.0135 -0.0198 -0.0158 -0.0079
Feature Noise     0.0646  0.0689  0.0624  0.0666  0.0425  0.0631  0.0614
Label Noise       0.9604  0.9204  0.9509  0.9568  0.9357  0.9788  0.9505
Missing Features  0.0884  0.1213  0.1277  0.1011  0.0958  0.0909  0.1042

  alpha COEFFICIENTS - metric: balanced_accuracy
algorithm             GB      LR     MLP      RF     SVM     XGB    Mean
dim_name                                                                
Class Imbalance   0.0902  0.0706  0.1628  0.0859  0.0775  0.0904  0.0962
Feature Noise     0.0833  0.0763  0.0681  0.0894  0.0493  0.0896  0.0760
Label Noise       0.8805  0.8342  0.8502  0.8496  0.8241  0.8996  0.8564
Missing Features  0.1188  0.1495  0.1525  0.1272  0.1196  0.1299  0.1329

  alpha COEFFICIENTS - metric: m

## 10. Phase C — Pairwise Interactions

**Crash-safe**: saves after each dimension pair.  
6 pairs × 4 severity combos × 5 datasets × 6 algorithms × 2 seeds = 1,440 experiments.

In [12]:
CP_NAME_C = 'phase_c_pairwise'
pairwise_existing = load_checkpoint(CP_NAME_C)

if pairwise_existing is not None and len(pairwise_existing) > 0:
    pw_results = pairwise_existing.to_dict('records')
    done_pairs = set()
    for r in pw_results:
        done_pairs.add((r.get('dim1',''), r.get('dim2',''),
                        r.get('sev1',0), r.get('sev2',0),
                        r['dataset'], r['algorithm'], r.get('seed',0)))
else:
    pw_results = []
    done_pairs = set()

dim_pairs = list(combinations(DIMENSIONS.keys(), 2))
total_expected = 0
for d1, d2 in dim_pairs:
    for s1 in DIMENSIONS[d1]['levels_pair']:
        for s2 in DIMENSIONS[d2]['levels_pair']:
            total_expected += len(DATASETS) * len(ALGORITHMS) * N_SEEDS_C

remaining = total_expected - len(done_pairs)

if remaining > 0:
    print("="*60)
    print(f"  PHASE C: PAIRWISE INTERACTIONS ({remaining} remaining of {total_expected})")
    print("="*60)

    with Timer():
        batch = []
        with tqdm(total=remaining, desc="Phase C") as pbar:
            for dim1, dim2 in dim_pairs:
                pair_batch = []
                for sev1 in DIMENSIONS[dim1]['levels_pair']:
                    for sev2 in DIMENSIONS[dim2]['levels_pair']:
                        profile = {d: 0 for d in DIMENSIONS}
                        profile[dim1] = sev1
                        profile[dim2] = sev2

                        for ds_key, dataset in DATASETS.items():
                            for algo_key, algo_info in ALGORITHMS.items():
                                for s in range(N_SEEDS_C):
                                    key = (dim1, dim2, sev1, sev2,
                                           dataset.name, algo_key, RANDOM_SEED+s)
                                    if key in done_pairs:
                                        continue
                                    r = run_experiment(dataset, algo_key, algo_info,
                                                      profile, seed=RANDOM_SEED+s)
                                    if r:
                                        r['dim1'] = dim1
                                        r['dim2'] = dim2
                                        r['sev1'] = sev1
                                        r['sev2'] = sev2
                                        pair_batch.append(r)
                                        pw_results.append(r)
                                    pbar.update(1)

                # Save after each dimension pair
                if pair_batch:
                    append_checkpoint(pair_batch, CP_NAME_C)
                    gc.collect()

pairwise = pd.DataFrame(pw_results)
save_checkpoint(pairwise, CP_NAME_C)
pairwise = add_degradation(pairwise, baselines)
print(f"\nPhase C complete: {len(pairwise)} experiments")

  PHASE C: PAIRWISE INTERACTIONS (1440 remaining of 1440)


Phase C: 100%|██████████| 1440/1440 [11:02<00:00,  2.17it/s] 


  Elapsed: 11m 2.5s

Phase C complete: 1440 experiments


In [13]:
def compute_interactions(pairwise_df, alpha_df):
    alpha_lk = {}
    for _, row in alpha_df.iterrows():
        alpha_lk[(row['algorithm'], row['dimension'], row['metric'])] = row['alpha']

    records = []
    for _, row in pairwise_df.iterrows():
        algo = row['algorithm']
        for metric in REPORT_METRICS:
            deg_col = f'deg_{metric}'
            if deg_col not in row or pd.isna(row[deg_col]):
                continue
            a1 = alpha_lk.get((algo, row['dim1'], metric), 0)
            a2 = alpha_lk.get((algo, row['dim2'], metric), 0)
            add_pred = a1 * row['sev1'] + a2 * row['sev2']
            obs_deg = row[deg_col]
            interaction = obs_deg - add_pred
            denom = row['sev1'] * row['sev2']
            gamma = interaction / denom if denom > 1e-10 else 0.0
            records.append({
                'algorithm': algo, 'dataset': row['dataset'],
                'dim1': row['dim1'], 'dim2': row['dim2'],
                'sev1': row['sev1'], 'sev2': row['sev2'],
                'metric': metric, 'observed_deg': obs_deg,
                'additive_pred': add_pred, 'interaction': interaction,
                'gamma': gamma,
            })
    return pd.DataFrame(records)

interactions = compute_interactions(pairwise, alpha_df)
save_checkpoint(interactions, 'interactions')
print(f"Interaction effects: {len(interactions)} records")

Interaction effects: 5760 records


## 11. Interaction Classification & Threshold Sensitivity (R2-D5)

In [14]:
def classify_interactions(interactions_df, metric='accuracy'):
    sub = interactions_df[interactions_df['metric'] == metric].copy()
    print(f"\n{'='*70}")
    print(f"  INTERACTION CLASSIFICATION - {metric}")
    print(f"{'='*70}")

    # Threshold sensitivity
    print("\n  (a) Threshold sensitivity:")
    rows = []
    for tau in INTERACTION_THRESHOLDS:
        total = len(sub)
        n_add = (sub['interaction'].abs() <= tau).sum()
        n_sup = (sub['interaction'] > tau).sum()
        n_sub = (sub['interaction'] < -tau).sum()
        rows.append([tau, n_add, f"{100*n_add/total:.1f}%",
                     n_sup, f"{100*n_sup/total:.1f}%",
                     n_sub, f"{100*n_sub/total:.1f}%"])
    print(tabulate(rows, headers=['tau','Add','%','Super','%','Sub','%'], tablefmt='grid'))

    # Per-pair t-test + effect size
    print("\n  (b) Per-pair statistical tests (t-test + Cohen's d):")
    pair_results = []
    for (d1, d2), grp in sub.groupby(['dim1','dim2']):
        vals = grp['interaction'].values
        if len(vals) < 3:
            continue
        t_stat, p_val = stats.ttest_1samp(vals, 0)
        mean_int = np.mean(vals)
        cohens_d = mean_int / (np.std(vals) + 1e-10)
        sig = '***' if p_val<0.001 else ('**' if p_val<0.01 else ('*' if p_val<0.05 else 'ns'))
        ptype = ('superadditive' if mean_int > 0 and p_val < 0.05
                 else ('subadditive' if mean_int < 0 and p_val < 0.05 else 'additive'))
        pair_results.append({
            'dim1': DIMENSIONS[d1]['short'], 'dim2': DIMENSIONS[d2]['short'],
            'mean_gamma': round(mean_int, 5), 'cohens_d': round(cohens_d, 3),
            't': round(t_stat, 2), 'p': round(p_val, 5), 'sig': sig, 'type': ptype
        })
    pdf = pd.DataFrame(pair_results)
    print(tabulate(pdf, headers='keys', tablefmt='grid', showindex=False))
    return pdf

int_results = {}
for metric in REPORT_METRICS:
    int_results[metric] = classify_interactions(interactions, metric)


  INTERACTION CLASSIFICATION - accuracy

  (a) Threshold sensitivity:
+-------+-------+-------+---------+-------+-------+-------+
|   tau |   Add | %     |   Super | %     |   Sub | %     |
+=======+=======+=======+=========+=======+=======+=======+
| 0.005 |   194 | 13.5% |     631 | 43.8% |   615 | 42.7% |
+-------+-------+-------+---------+-------+-------+-------+
| 0.01  |   343 | 23.8% |     553 | 38.4% |   544 | 37.8% |
+-------+-------+-------+---------+-------+-------+-------+
| 0.02  |   644 | 44.7% |     369 | 25.6% |   427 | 29.7% |
+-------+-------+-------+---------+-------+-------+-------+
| 0.05  |  1142 | 79.3% |     112 | 7.8%  |   186 | 12.9% |
+-------+-------+-------+---------+-------+-------+-------+

  (b) Per-pair statistical tests (t-test + Cohen's d):
+--------+--------+--------------+------------+-------+---------+-------+---------------+
| dim1   | dim2   |   mean_gamma |   cohens_d |     t |       p | sig   | type          |
+========+========+==============

## 12. Composition Model Fitting

- **M1 Additive**: Delta = sum(alpha_j * s_j)
- **M2 Interaction**: M1 + sum(gamma_ij * s_i * s_j)
- **M3 Polynomial**: M1 + sum(beta_j * s_j^2) + sum(gamma_ij * s_i * s_j)

M3 = second-order Taylor expansion of arbitrary smooth degradation function (R3-W3).

In [15]:
def prepare_model_data(single_df, pairwise_df, metric='accuracy'):
    dim_names = list(DIMENSIONS.keys())
    deg_col = f'deg_{metric}'
    combined = pd.concat([single_df, pairwise_df], ignore_index=True)
    profiles, targets = [], []
    for _, row in combined.iterrows():
        profile = [row.get(d, 0) for d in dim_names]
        if deg_col in row and not pd.isna(row[deg_col]):
            profiles.append(profile)
            targets.append(row[deg_col])
    return np.array(profiles), np.array(targets)

def fit_composition_models(X, y, test_size=0.2, seed=RANDOM_SEED):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=test_size, random_state=seed)
    n_dims = X.shape[1]

    def _interact(Z):
        parts = [Z]
        for i in range(n_dims):
            for j in range(i+1, n_dims):
                parts.append((Z[:,i]*Z[:,j]).reshape(-1,1))
        return np.hstack(parts)

    def _poly(Z):
        parts = [Z, Z**2]
        for i in range(n_dims):
            for j in range(i+1, n_dims):
                parts.append((Z[:,i]*Z[:,j]).reshape(-1,1))
        return np.hstack(parts)

    results = []
    for name, tfn in [('M1_Additive', lambda Z: Z),
                      ('M2_Interaction', _interact),
                      ('M3_Polynomial', _poly)]:
        Xtr_t, Xte_t = tfn(X_tr), tfn(X_te)
        model = Ridge(alpha=0.01, fit_intercept=False).fit(Xtr_t, y_tr)
        p_tr, p_te = model.predict(Xtr_t), model.predict(Xte_t)

        r2_tr = 1 - np.sum((y_tr-p_tr)**2)/(np.sum((y_tr-np.mean(y_tr))**2)+1e-12)
        r2_te = 1 - np.sum((y_te-p_te)**2)/(np.sum((y_te-np.mean(y_te))**2)+1e-12)
        rmse = np.sqrt(mean_squared_error(y_te, p_te))
        mae = mean_absolute_error(y_te, p_te)
        k = len(model.coef_)
        n = len(y_te)
        sse = np.sum((y_te-p_te)**2)+1e-12
        aic = n*np.log(sse/n) + 2*k
        bic = n*np.log(sse/n) + k*np.log(n)
        results.append({'Model': name, 'Params': k,
                        'Train_R2': r2_tr, 'Test_R2': r2_te,
                        'RMSE': rmse, 'MAE': mae, 'AIC': aic, 'BIC': bic})
    return pd.DataFrame(results)

model_results = {}
for metric in REPORT_METRICS:
    X_m, y_m = prepare_model_data(single_dim, pairwise, metric)
    res = fit_composition_models(X_m, y_m)
    model_results[metric] = res
    print(f"\n{'='*70}")
    print(f"  COMPOSITION MODELS - {metric}")
    print(f"{'='*70}")
    print(tabulate(res, headers='keys', tablefmt='grid', showindex=False, floatfmt='.4f'))

save_checkpoint(model_results['accuracy'], 'model_comparison')


  COMPOSITION MODELS - accuracy
+----------------+----------+------------+-----------+--------+--------+------------+------------+
| Model          |   Params |   Train_R2 |   Test_R2 |   RMSE |    MAE |        AIC |        BIC |
+================+==========+============+===========+========+========+============+============+
| M1_Additive    |        4 |     0.9015 |    0.9242 | 0.0390 | 0.0266 | -3728.4709 | -3711.0465 |
+----------------+----------+------------+-----------+--------+--------+------------+------------+
| M2_Interaction |       10 |     0.9090 |    0.9247 | 0.0389 | 0.0263 | -3720.1783 | -3676.6172 |
+----------------+----------+------------+-----------+--------+--------+------------+------------+
| M3_Polynomial  |       14 |     0.9109 |    0.9266 | 0.0384 | 0.0258 | -3726.9343 | -3665.9488 |
+----------------+----------+------------+-----------+--------+--------+------------+------------+

  COMPOSITION MODELS - balanced_accuracy
+----------------+----------+-----

'./cdqc_final_checkpoints/model_comparison.csv'

## 13. Ablation Study

In [16]:
def run_ablation(X, y, dim_names):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)
    def _r2(Xtr, Xte):
        m = Ridge(alpha=0.01, fit_intercept=False).fit(Xtr, y_tr)
        p = m.predict(Xte)
        return 1 - np.sum((y_te-p)**2)/(np.sum((y_te-np.mean(y_te))**2)+1e-12)

    full_r2 = _r2(X_tr, X_te)
    rows = [{'Config': 'Full model', 'Test_R2': full_r2, 'Delta_R2': 0}]

    for i, dim in enumerate(dim_names):
        keep = [j for j in range(len(dim_names)) if j != i]
        r2 = _r2(X_tr[:, keep], X_te[:, keep])
        rows.append({'Config': f'Without {DIMENSIONS[dim]["name"]}',
                     'Test_R2': r2, 'Delta_R2': full_r2 - r2})

    for i, dim in enumerate(dim_names):
        r2 = _r2(X_tr[:, [i]], X_te[:, [i]])
        rows.append({'Config': f'Only {DIMENSIONS[dim]["name"]}',
                     'Test_R2': r2, 'Delta_R2': full_r2 - r2})
    return pd.DataFrame(rows)

ablation_results = {}
for metric in REPORT_METRICS:
    X_m, y_m = prepare_model_data(single_dim, pairwise, metric)
    abl = run_ablation(X_m, y_m, list(DIMENSIONS.keys()))
    ablation_results[metric] = abl
    print(f"\n  ABLATION - {metric}:")
    print(tabulate(abl, headers='keys', tablefmt='grid', showindex=False, floatfmt='.4f'))


  ABLATION - accuracy:
+--------------------------+-----------+------------+
| Config                   |   Test_R2 |   Delta_R2 |
+==========================+===========+============+
| Full model               |    0.9242 |     0.0000 |
+--------------------------+-----------+------------+
| Without Label Noise      |   -0.3779 |     1.3022 |
+--------------------------+-----------+------------+
| Without Missing Features |    0.9193 |     0.0049 |
+--------------------------+-----------+------------+
| Without Feature Noise    |    0.9161 |     0.0082 |
+--------------------------+-----------+------------+
| Without Class Imbalance  |    0.9211 |     0.0031 |
+--------------------------+-----------+------------+
| Only Label Noise         |    0.9082 |     0.0161 |
+--------------------------+-----------+------------+
| Only Missing Features    |   -0.4529 |     1.3771 |
+--------------------------+-----------+------------+
| Only Feature Noise       |   -0.4008 |     1.3250 |
+---

## 14. Statistical Significance (Friedman + Nemenyi + Bootstrap)

In [17]:
def run_statistical_tests(alpha_df, interactions_df, metric='accuracy'):
    print(f"\n{'='*70}")
    print(f"  STATISTICAL TESTS - {metric}")
    print(f"{'='*70}")

    sub_a = alpha_df[alpha_df['metric'] == metric]
    sub_i = interactions_df[interactions_df['metric'] == metric]
    dims = sorted(DIMENSIONS.keys())
    algos = sorted(sub_a['algorithm'].unique())

    # 1. Friedman test
    matrix = []
    for algo in algos:
        row = []
        for dim in dims:
            vals = sub_a[(sub_a['algorithm']==algo) & (sub_a['dimension']==dim)]['alpha'].values
            row.append(vals[0] if len(vals)>0 else 0)
        matrix.append(row)
    matrix = np.array(matrix)

    try:
        chi2, p = stats.friedmanchisquare(*[matrix[:,i] for i in range(matrix.shape[1])])
        print(f"\n  Friedman: chi2={chi2:.2f}, p={p:.6f}, df={len(dims)-1}")
    except Exception as e:
        print(f"  Friedman failed: {e}")
        chi2, p = np.nan, np.nan

    # 2. Nemenyi post-hoc
    try:
        nem = sp.posthoc_nemenyi_friedman(matrix)
        nem.index = [DIMENSIONS[d]['short'] for d in dims]
        nem.columns = nem.index
        print(f"\n  Nemenyi post-hoc p-values:")
        print(nem.round(4).to_string())
    except Exception as e:
        print(f"  Nemenyi failed: {e}")

    # 3. Bootstrap CIs
    print(f"\n  Bootstrap CIs (n=1000):")
    for dim in dims:
        vals = sub_a[sub_a['dimension']==dim]['alpha'].values
        if len(vals) >= 2:
            ci = bootstrap_ci(vals, 1000)
            print(f"    alpha_{DIMENSIONS[dim]['short']:2s} = {np.mean(vals):.4f}  "
                  f"95% CI [{ci[0]:.4f}, {ci[1]:.4f}]")

for metric in REPORT_METRICS:
    run_statistical_tests(alpha_df, interactions, metric)


  STATISTICAL TESTS - accuracy

  Friedman: chi2=18.00, p=0.000440, df=3

  Nemenyi post-hoc p-values:
        FN      CI      LN      MF
FN  1.0000  0.5363  0.0366  0.5363
CI  0.5363  1.0000  0.0003  0.0366
LN  0.0366  0.0003  1.0000  0.5363
MF  0.5363  0.0366  0.5363  1.0000

  Bootstrap CIs (n=1000):
    alpha_FN = 0.0614  95% CI [0.0535, 0.0664]
    alpha_CI = -0.0079  95% CI [-0.0205, 0.0130]
    alpha_LN = 0.9505  95% CI [0.9363, 0.9643]
    alpha_MF = 0.1042  95% CI [0.0930, 0.1161]

  STATISTICAL TESTS - balanced_accuracy

  Friedman: chi2=15.00, p=0.001817, df=3

  Nemenyi post-hoc p-values:
        FN      CI      LN      MF
FN  1.0000  0.9081  0.0020  0.1833
CI  0.9081  1.0000  0.0191  0.5363
LN  0.0020  0.0191  1.0000  0.3986
MF  0.1833  0.5363  0.3986  1.0000

  Bootstrap CIs (n=1000):
    alpha_FN = 0.0760  95% CI [0.0648, 0.0852]
    alpha_CI = 0.0962  95% CI [0.0787, 0.1232]
    alpha_LN = 0.8564  95% CI [0.8385, 0.8766]
    alpha_MF = 0.1329  95% CI [0.1223, 0.1435]



## 15. Phase D — Extreme Severity Validation (R1-D5, Theorem 3)

In [18]:
extreme = load_checkpoint('phase_d_extreme')

if extreme is None:
    print("="*60)
    print("  PHASE D: EXTREME SEVERITY VALIDATION")
    print("="*60)
    results = []
    extreme_levels = [0.50, 0.60, 0.70]

    with Timer():
        # Single-dimension extreme
        for dim_name in DIMENSIONS.keys():
            for sev in extreme_levels:
                profile = {d: 0 for d in DIMENSIONS}
                profile[dim_name] = sev
                for ds_key, dataset in DATASETS.items():
                    for algo_key, algo_info in ALGORITHMS.items():
                        r = run_experiment(dataset, algo_key, algo_info,
                                          profile, seed=RANDOM_SEED)
                        if r:
                            r['active_dim'] = dim_name
                            r['severity'] = sev
                            r['phase'] = 'extreme_single'
                            results.append(r)

        # Pairwise extreme: LN x MF
        for s1 in extreme_levels:
            for s2 in extreme_levels:
                profile = {d: 0 for d in DIMENSIONS}
                profile['label_noise'] = s1
                profile['missing'] = s2
                for ds_key, dataset in DATASETS.items():
                    for algo_key, algo_info in ALGORITHMS.items():
                        r = run_experiment(dataset, algo_key, algo_info,
                                          profile, seed=RANDOM_SEED)
                        if r:
                            r['dim1'] = 'label_noise'
                            r['dim2'] = 'missing'
                            r['sev1'] = s1
                            r['sev2'] = s2
                            r['phase'] = 'extreme_pair'
                            results.append(r)

    extreme = pd.DataFrame(results)
    save_checkpoint(extreme, 'phase_d_extreme')

extreme = add_degradation(extreme, baselines)
print(f"Phase D: {len(extreme)} experiments")

# ── Validate additive bound at extreme severity ──
alpha_lk = {}
for _, row in alpha_df[alpha_df['metric']=='accuracy'].iterrows():
    alpha_lk[(row['algorithm'], row['dimension'])] = row['alpha']

ext_pair = extreme[extreme.get('phase','') == 'extreme_pair'] if 'phase' in extreme.columns else pd.DataFrame()
if len(ext_pair) > 0:
    errors = []
    for _, row in ext_pair.iterrows():
        a1 = alpha_lk.get((row['algorithm'], row.get('dim1','')), 0)
        a2 = alpha_lk.get((row['algorithm'], row.get('dim2','')), 0)
        pred = a1*row.get('sev1',0) + a2*row.get('sev2',0)
        errors.append(abs(row['deg_accuracy'] - pred))
    errors = np.array(errors)
    smax, k, eps = 0.7, 4, DEFAULT_THRESHOLD
    bound = eps * (k*(k-1)/2) * smax**2
    print(f"\n  Additive error at extreme severity:")
    print(f"    Mean |error| = {np.mean(errors):.4f}")
    print(f"    Max  |error| = {np.max(errors):.4f}")
    print(f"    Theorem 3 bound (eps={eps}, smax={smax}): {bound:.4f}")
    print(f"    Fraction exceeding: {np.mean(errors>bound):.1%}")

  PHASE D: EXTREME SEVERITY VALIDATION
  Elapsed: 3m 25.7s
Phase D: 630 experiments

  Additive error at extreme severity:
    Mean |error| = 0.2104
    Max  |error| = 0.5180
    Theorem 3 bound (eps=0.01, smax=0.7): 0.0294
    Fraction exceeding: 96.7%


## 16. Phase E — Correlated Corruption Validation (R1-D2, R2-D3)

Tests whether the additive model (calibrated on independent corruptions)
still predicts accurately when corruptions are **correlated** — same
instances affected by multiple quality issues.

In [19]:
def apply_correlated_corruption(X, y, profile, seed=RANDOM_SEED):
    """Correlated corruption: same vulnerable instances get hit by multiple issues."""
    rng = np.random.RandomState(seed)
    X_new, y_new = X.copy(), y.copy()
    vuln = rng.random(len(y))  # shared vulnerability score

    # Label noise on most vulnerable
    ln_sev = profile.get('label_noise', 0)
    if ln_sev > 0:
        flip_mask = vuln < ln_sev
        classes = np.unique(y)
        for i in np.where(flip_mask)[0]:
            wrong = classes[classes != y_new[i]]
            y_new[i] = rng.choice(wrong)

    # Missing: vulnerable instances lose more features
    mf_sev = profile.get('missing', 0)
    if mf_sev > 0:
        miss_prob = np.where(vuln < max(ln_sev, 0.01), mf_sev*1.5, mf_sev*0.5)
        miss_prob = np.clip(miss_prob, 0, 1)
        col_medians = np.nanmedian(X_new, axis=0)
        for col in range(X_new.shape[1]):
            mask = rng.random(len(y)) < miss_prob
            X_new[mask, col] = col_medians[col]

    # Feature noise: correlated with vulnerability
    fn_sev = profile.get('feature_noise', 0)
    if fn_sev > 0:
        noise_scale = np.where(vuln < max(ln_sev, mf_sev, 0.01),
                               fn_sev*1.3, fn_sev*0.7)
        sigma_j = np.std(X_new, axis=0) + 1e-10
        for col in range(X_new.shape[1]):
            X_new[:, col] += rng.normal(0, noise_scale * sigma_j[col])

    return X_new, y_new

corr_results = load_checkpoint('phase_e_correlated')

if corr_results is None:
    print("="*60)
    print("  PHASE E: CORRELATED CORRUPTION VALIDATION")
    print("="*60)
    test_profiles = [
        {'label_noise': 0.15, 'missing': 0.10, 'feature_noise': 0, 'imbalance': 0},
        {'label_noise': 0.25, 'missing': 0.15, 'feature_noise': 0.10, 'imbalance': 0},
        {'label_noise': 0.35, 'missing': 0.20, 'feature_noise': 0.20, 'imbalance': 0},
        {'label_noise': 0.20, 'missing': 0.10, 'feature_noise': 0.15, 'imbalance': 0.30},
    ]
    records = []
    with Timer():
        for profile in test_profiles:
            for ds_key, dataset in DATASETS.items():
                for algo_key, algo_info in ALGORITHMS.items():
                    # Independent
                    X_ind, y_ind = dataset.X.copy(), dataset.y.copy()
                    for dim, sev in profile.items():
                        if sev > 0:
                            X_ind, y_ind = DIMENSIONS[dim]['fn'](X_ind, y_ind, sev, seed=RANDOM_SEED)
                    # Correlated
                    X_cor, y_cor = apply_correlated_corruption(dataset.X, dataset.y, profile)

                    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
                    for mode, Xm, ym in [('independent', X_ind, y_ind), ('correlated', X_cor, y_cor)]:
                        if len(ym) < N_FOLDS*3 or len(np.unique(ym)) < 2:
                            continue
                        try:
                            cv_res = cross_validate(clone(algo_info['model']), Xm, ym,
                                                    cv=cv, scoring=METRICS, n_jobs=-1)
                            rec = {'dataset': ds_key, 'algorithm': algo_key, 'mode': mode, **profile}
                            for m in REPORT_METRICS:
                                rec[m] = float(np.mean(cv_res[f'test_{m}']))
                            records.append(rec)
                        except:
                            pass

    corr_results = pd.DataFrame(records)
    save_checkpoint(corr_results, 'phase_e_correlated')

print(f"Phase E: {len(corr_results)} experiments")
print("\n  Independent vs Correlated comparison:")
for metric in REPORT_METRICS:
    ind = corr_results[corr_results['mode']=='independent'][metric].values
    cor = corr_results[corr_results['mode']=='correlated'][metric].values
    n = min(len(ind), len(cor))
    if n > 2:
        t_stat, p_val = stats.ttest_rel(ind[:n], cor[:n])
        print(f"    {metric:22s}: Delta={np.mean(ind[:n])-np.mean(cor[:n]):+.4f}  "
              f"t={t_stat:.2f}  p={p_val:.4f}")

  PHASE E: CORRELATED CORRUPTION VALIDATION
  Elapsed: 2m 25.1s
Phase E: 240 experiments

  Independent vs Correlated comparison:
    accuracy              : Delta=-0.0401  t=-7.48  p=0.0000
    balanced_accuracy     : Delta=-0.0580  t=-8.61  p=0.0000
    macro_f1              : Delta=-0.0653  t=-8.28  p=0.0000
    mcc                   : Delta=-0.1136  t=-8.50  p=0.0000


## 17. Publication-Quality Figures

In [20]:
COLORS_DIM = ['#e41a1c','#377eb8','#4daf4a','#984ea3']
COLORS_ALG = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd','#8c564b']

# ═══════════════════════════════════════════════════════════════
# Fig 1: Alpha coefficients by algorithm
# ═══════════════════════════════════════════════════════════════
sub = alpha_df[alpha_df['metric'] == 'accuracy']
pivot = sub.pivot_table(index='dim_name', columns='algorithm', values='alpha', aggfunc='mean')

fig, ax = plt.subplots(figsize=(10, 6))
pivot.plot(kind='bar', ax=ax, color=COLORS_ALG[:len(pivot.columns)])
ax.set_ylabel('Sensitivity Coefficient (alpha)')
ax.set_xlabel('Quality Dimension')
ax.set_title('Sensitivity Coefficients by Algorithm (Accuracy)')
ax.legend(title='Algorithm', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
save_fig(fig, 'fig1_alpha_by_algorithm')

# ═══════════════════════════════════════════════════════════════
# Fig 2: Degradation curves — accuracy vs balanced accuracy
# ═══════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for idx, (dim_key, dim_info) in enumerate(DIMENSIONS.items()):
    ax = axes[idx//2][idx%2]
    sub = single_dim[single_dim['active_dim'] == dim_key]
    for mi, metric in enumerate(['accuracy', 'balanced_accuracy']):
        deg_col = f'deg_{metric}'
        agg = sub.groupby('severity')[deg_col].agg(['mean','std']).reset_index()
        color = COLORS_DIM[idx] if mi == 0 else 'gray'
        ls = '-' if mi == 0 else '--'
        ax.errorbar(agg['severity'], agg['mean'], yerr=agg['std'],
                    fmt='o-', color=color, ls=ls, capsize=3,
                    label=metric.replace('_',' ').title(), markersize=5)
    ax.set_title(dim_info['name'], fontsize=13)
    ax.set_xlabel('Severity')
    ax.set_ylabel('Degradation')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
plt.suptitle('Degradation Curves: Accuracy vs Balanced Accuracy', fontsize=14, y=1.01)
plt.tight_layout()
save_fig(fig, 'fig2_degradation_curves')

# ═══════════════════════════════════════════════════════════════
# Fig 3: Multi-metric alpha comparison (KEY NEW FIGURE)
# ═══════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(12, 6))
mam = []
for metric in REPORT_METRICS:
    sub = alpha_df[alpha_df['metric'] == metric]
    for dim_name, val in sub.groupby('dim_name')['alpha'].mean().items():
        mam.append({'Metric': metric.replace('_',' ').title(), 'Dimension': dim_name, 'Alpha': val})
mam_df = pd.DataFrame(mam)
mam_pivot = mam_df.pivot(index='Dimension', columns='Metric', values='Alpha')
mam_pivot.plot(kind='bar', ax=ax, width=0.7)
ax.set_ylabel('Mean Sensitivity Coefficient (alpha)')
ax.set_title('How Sensitivity Changes Across Metrics (R1-D1 Response)')
ax.legend(title='Metric', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
save_fig(fig, 'fig3_multimetric_alpha')

# ═══════════════════════════════════════════════════════════════
# Fig 4: Interaction heatmap (accuracy + balanced accuracy)
# ═══════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for mi, metric in enumerate(['accuracy', 'balanced_accuracy']):
    ax = axes[mi]
    sub = interactions[interactions['metric'] == metric]
    int_agg = sub.groupby(['dim1','dim2'])['interaction'].mean().reset_index()
    dims_list = list(DIMENSIONS.keys())
    n = len(dims_list)
    mat = np.zeros((n, n))
    for _, row in int_agg.iterrows():
        i, j = dims_list.index(row['dim1']), dims_list.index(row['dim2'])
        mat[i,j] = row['interaction']; mat[j,i] = row['interaction']
    labels = [DIMENSIONS[d]['short'] for d in dims_list]
    sns.heatmap(mat, xticklabels=labels, yticklabels=labels,
                center=0, cmap='RdBu_r', annot=True, fmt='.4f',
                ax=ax, square=True, cbar_kws={'label': 'Interaction'})
    ax.set_title(f'Interactions - {metric}')
plt.tight_layout()
save_fig(fig, 'fig4_interaction_heatmap')

# ═══════════════════════════════════════════════════════════════
# Fig 5: Composition model comparison
# ═══════════════════════════════════════════════════════════════
res = model_results['accuracy']
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
x = np.arange(len(res)); w = 0.35
ax = axes[0]
ax.bar(x-w/2, res['Train_R2'], w, label='Train R2', color='#1b9e77', alpha=0.8)
ax.bar(x+w/2, res['Test_R2'], w, label='Test R2', color='#d95f02', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(res['Model'])
ax.set_ylabel('R2'); ax.set_title('Predictive Performance'); ax.legend()
for i, row in res.iterrows():
    ax.annotate(f"{row['Test_R2']:.3f}", (i+w/2, row['Test_R2']+0.005), ha='center', fontsize=9)
ax = axes[1]
ax.scatter(res['Params'], res['Test_R2'], s=120, c=['#1b9e77','#d95f02','#7570b3'], zorder=5)
for _, row in res.iterrows():
    ax.annotate(f"  {row['Model']}", (row['Params'], row['Test_R2']), fontsize=10)
ax.set_xlabel('Parameters'); ax.set_ylabel('Test R2'); ax.set_title('Parsimony vs Performance')
plt.tight_layout()
save_fig(fig, 'fig5_model_comparison')

# ═══════════════════════════════════════════════════════════════
# Fig 6: Ablation
# ═══════════════════════════════════════════════════════════════
abl = ablation_results['accuracy']
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
loo = abl[abl['Config'].str.startswith('Without')].sort_values('Delta_R2', ascending=False)
axes[0].barh(loo['Config'], loo['Delta_R2'], color=COLORS_DIM[:len(loo)])
axes[0].set_xlabel('Delta R2 (drop when removed)'); axes[0].set_title('Leave-One-Out')
only = abl[abl['Config'].str.startswith('Only')].sort_values('Test_R2', ascending=False)
axes[1].barh(only['Config'], only['Test_R2'], color=COLORS_DIM[:len(only)])
axes[1].set_xlabel('Test R2'); axes[1].set_title('Single-Dimension Explanatory Power')
axes[1].axvline(x=abl.iloc[0]['Test_R2'], color='red', ls='--', label='Full model'); axes[1].legend()
plt.tight_layout()
save_fig(fig, 'fig6_ablation')

# ═══════════════════════════════════════════════════════════════
# Fig 7: Threshold sensitivity (R2-D5)
# ═══════════════════════════════════════════════════════════════
sub = interactions[interactions['metric'] == 'accuracy']
fig, ax = plt.subplots(figsize=(10, 5))
tau_data = []
for tau in INTERACTION_THRESHOLDS:
    total = len(sub)
    tau_data.append({'tau': str(tau),
        'Additive': 100*(sub['interaction'].abs()<=tau).sum()/total,
        'Superadditive': 100*(sub['interaction']>tau).sum()/total,
        'Subadditive': 100*(sub['interaction']<-tau).sum()/total})
tdf = pd.DataFrame(tau_data)
x = np.arange(len(tdf)); w = 0.25
ax.bar(x-w, tdf['Additive'], w, label='Additive', color='#2ca02c')
ax.bar(x, tdf['Superadditive'], w, label='Superadditive', color='#d62728')
ax.bar(x+w, tdf['Subadditive'], w, label='Subadditive', color='#1f77b4')
ax.set_xticks(x); ax.set_xticklabels([f"tau={t}" for t in tdf['tau']])
ax.set_ylabel('% of Pairs'); ax.set_title('Threshold Sensitivity (R2-D5)'); ax.legend()
plt.tight_layout()
save_fig(fig, 'fig7_threshold_sensitivity')

print("\nAll 7 figures saved to:", FIGURE_DIR)

  Saved figure: fig1_alpha_by_algorithm
  Saved figure: fig2_degradation_curves
  Saved figure: fig3_multimetric_alpha
  Saved figure: fig4_interaction_heatmap
  Saved figure: fig5_model_comparison
  Saved figure: fig6_ablation
  Saved figure: fig7_threshold_sensitivity

All 7 figures saved to: ./cdqc_final_figures


## 18. Final Summary & Export

In [21]:
print("\n" + "="*80)
print("        CDQC FINAL EXPERIMENTAL RESULTS")
print("="*80)

n_bl = len(baselines)
n_sd = len(single_dim)
n_pw = len(pairwise)
n_ex = len(extreme) if extreme is not None else 0
n_co = len(corr_results) if corr_results is not None else 0
total = n_bl + n_sd + n_pw + n_ex + n_co
print(f"\n  Total experiments: {total}")
print(f"    Phase A (Baselines):            {n_bl:>6}")
print(f"    Phase B (Single-dimension):     {n_sd:>6}")
print(f"    Phase C (Pairwise):             {n_pw:>6}")
print(f"    Phase D (Extreme):              {n_ex:>6}")
print(f"    Phase E (Correlated):           {n_co:>6}")

print(f"\n  Datasets: {list(DATASETS.keys())}")
print(f"  Algorithms: {list(ALGORITHMS.keys())}")
print(f"  Metrics: {REPORT_METRICS}")

# Dominance hierarchy per metric
print(f"\n  DOMINANCE HIERARCHY:")
for metric in REPORT_METRICS:
    sub = alpha_df[alpha_df['metric'] == metric]
    means = sub.groupby('dim_name')['alpha'].mean().sort_values(ascending=False)
    ranking = ' > '.join([f"{d}({v:.3f})" for d, v in means.items()])
    print(f"    {metric:22s}: {ranking}")

# Practitioner formula
print(f"\n  PRACTITIONER FORMULAS:")
for metric in ['accuracy', 'balanced_accuracy']:
    sub = alpha_df[alpha_df['metric'] == metric]
    means = sub.groupby('dim_short')['alpha'].mean()
    terms = [f"{means.get(s,0):.3f}*s_{s}" for s in ['LN','MF','FN','CI']]
    print(f"    {metric}: Delta = {' + '.join(terms)}")

# Composition model
for metric in ['accuracy', 'balanced_accuracy']:
    res = model_results[metric]
    m1 = res[res['Model']=='M1_Additive'].iloc[0]
    print(f"\n  M1 Additive ({metric}): Test R2={m1['Test_R2']:.4f}  RMSE={m1['RMSE']:.4f}")

print(f"\n  Checkpoints: {CHECKPOINT_DIR}/")
print(f"  Figures:     {FIGURE_DIR}/")


        CDQC FINAL EXPERIMENTAL RESULTS

  Total experiments: 3840
    Phase A (Baselines):                90
    Phase B (Single-dimension):       1440
    Phase C (Pairwise):               1440
    Phase D (Extreme):                 630
    Phase E (Correlated):              240

  Datasets: ['spambase', 'bank', 'wine', 'breast_cancer', 'digits']
  Algorithms: ['LR', 'RF', 'XGB', 'SVM', 'MLP', 'GB']
  Metrics: ['accuracy', 'balanced_accuracy', 'macro_f1', 'mcc']

  DOMINANCE HIERARCHY:
    accuracy              : Label Noise(0.950) > Missing Features(0.104) > Feature Noise(0.061) > Class Imbalance(-0.008)
    balanced_accuracy     : Label Noise(0.856) > Missing Features(0.133) > Class Imbalance(0.096) > Feature Noise(0.076)
    macro_f1              : Label Noise(0.894) > Missing Features(0.140) > Class Imbalance(0.090) > Feature Noise(0.076)
    mcc                   : Label Noise(1.637) > Missing Features(0.245) > Class Imbalance(0.155) > Feature Noise(0.138)

  PRACTITIONER FORMU

In [22]:
# Save final consolidated artifacts
save_checkpoint(baselines, 'FINAL_baselines')
save_checkpoint(single_dim, 'FINAL_single_dimension')
save_checkpoint(pairwise, 'FINAL_pairwise')
save_checkpoint(alpha_df, 'FINAL_alpha_coefficients')
save_checkpoint(interactions, 'FINAL_interactions')
if extreme is not None:
    save_checkpoint(extreme, 'FINAL_extreme')
if corr_results is not None:
    save_checkpoint(corr_results, 'FINAL_correlated')
for metric in REPORT_METRICS:
    save_checkpoint(model_results[metric], f'FINAL_models_{metric}')
    save_checkpoint(ablation_results[metric], f'FINAL_ablation_{metric}')

config = {
    'random_seed': RANDOM_SEED, 'max_samples': MAX_SAMPLES,
    'n_folds': N_FOLDS, 'n_seeds_ab': N_SEEDS_AB, 'n_seeds_c': N_SEEDS_C,
    'metrics': REPORT_METRICS,
    'datasets': list(DATASETS.keys()),
    'algorithms': list(ALGORITHMS.keys()),
    'dimensions': {k: {'levels_single': v['levels_single'],
                        'levels_pair': v['levels_pair']}
                   for k, v in DIMENSIONS.items()},
}
with open(f"{CHECKPOINT_DIR}/experiment_config.json", 'w') as f:
    json.dump(config, f, indent=2)

print("All artifacts saved. Notebook complete.")

All artifacts saved. Notebook complete.


---

## Experiment Complete

### Reviewer Concerns Addressed

| # | Concern | Section | Evidence |
|---|---------|---------|----------|
| R1-D1 | Metric dependence of alphaCI | Sec 9, Fig 3 | 4 metrics; alphaCI shifts from negative (acc) to positive (bal_acc, F1, MCC) |
| R1-D2 | Synthetic-only validation | Sec 16 | Correlated corruption profiles tested |
| R1-D3 | Reproducibility | Sec 18 | Full config JSON + all checkpoints |
| R1-D5 | Theorem 3 at extreme severity | Sec 15 | Validation at s=0.5, 0.6, 0.7 |
| R2-D3 | Real-world validity | Sec 16 | Same-instance correlated corruption |
| R2-D5 | Arbitrary threshold | Sec 11, Fig 7 | Sensitivity across 4 thresholds + t-test + Cohen's d |
| R2-D6 | Scope limitations | Sec 9 | Multi-metric + multi-domain analysis |
| R3-W1 | Figure quality | Sec 17 | 300 DPI, proper fonts, clear labels |
| R3-W2 | Interactions matter but don't | Sec 9, 11 | Metric-dependent: interactions may differ under bal_acc |
| R3-W3 | M3 justification | Sec 12 | Documented as 2nd-order Taylor expansion |

### Bug Fixes Applied

| Bug | Fix | Impact |
|-----|-----|--------|
| Label noise sampled from all classes | Now samples from Y\\{yi} only | Higher effective noise → higher alphaLN |
| Class imbalance severity inverted | Higher = more imbalance | alphaCI now meaningful |
| Threshold mismatch (code=0.02, paper=0.01) | Unified at 0.01 + sensitivity analysis | Consistent reporting |
| Adult dataset encoding failure | Replaced with Spambase | Clean baselines across all datasets |

---